# Qlora微調對話式ChatGPT語言模型

Finetune Dialog LLM

使用Langbot bloom裁切小模型做展示 ('YeungNLP/bloomz-396m-zh'小模型也可以，過程一樣)

Colab T4 可以在不到兩小時之內訓練一個epoch成功! P100 GPU也OK!

只要一個epoch效果就很不錯!

In [4]:
!nvidia-smi

'nvidia-smi' ���O�����Υ~���R�O�B�i���檺�{���Χ妸�ɡC


# Install packages

In [5]:
!pip install transformers==4.32
!pip install datasets
!pip install peft
!pip install accelerate
!pip install bitsandbytes

In [6]:
# from google.colab import drive
# drive.mount('/content/drive')

In [7]:
cd 

C:\Users\USER


In [8]:
import os
import sys
import torch
import transformers
from transformers import Trainer, TrainingArguments
from transformers import BloomForCausalLM, BloomTokenizerFast,AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from peft import (
    prepare_model_for_int8_training,
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    prepare_model_for_kbit_training,
)

import datasets
import bitsandbytes
import peft

'NoneType' object has no attribute 'cadam32bit_grad_fp32'


c:\Users\USER\miniconda3\envs\ai23\lib\site-packages\bitsandbytes\cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


# Load model and tokenizer


    第2種小模型: 原始模型式float16，模型更小 
    tokenizer = BloomTokenizerFast.from_pretrained('YeungNLP/bloomz-396m-zh')
    第2種小模型: 原始模型式float16，模型更小 
    model = AutoModelForCausalLM.from_pretrained('YeungNLP/bloomz-396m-zh',torch_dtype='auto')
    資料集必須使用相同的tokenizer先處理。參看資料集前處理步驟之程式碼!

In [9]:
tokenizer = BloomTokenizerFast.from_pretrained('YeungNLP/bloomz-396m-zh')

In [10]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'pad_token': '<pad>'}

## Load base model

In [12]:
model_name = 'YeungNLP/bloomz-396m-zh' #float16  699MB
device_map='auto'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device_map,
    load_in_4bit=True,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        llm_int8_threshold=6.0,
        llm_int8_has_fp16_weight=False,
    ),
)

ImportError: Using `load_in_8bit=True` requires Accelerate: `pip install accelerate` and the latest version of bitsandbytes `pip install -i https://test.pypi.org/simple/ bitsandbytes` or pip install bitsandbytes` 

In [ ]:
model.config


BloomConfig {
  "_name_or_path": "YeungNLP/bloomz-396m-zh",
  "apply_residual_connection_post_layernorm": false,
  "architectures": [
    "BloomForCausalLM"
  ],
  "attention_dropout": 0.0,
  "attention_softmax_in_fp32": true,
  "bias_dropout_fusion": true,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_dropout": 0.0,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "masked_softmax_fusion": true,
  "model_type": "bloom",
  "n_head": 16,
  "n_inner": null,
  "n_layer": 24,
  "offset_alibi": 100,
  "pad_token_id": 3,
  "pretraining_tp": 1,
  "quantization_config": {
    "bnb_4bit_compute_dtype": "float16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "seq_length": 2048,
  "skip

In [ ]:
# casts all the non int8 modules to full precision (fp32) for stability
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

In [ ]:
print(f'memory footprint of model: {model.get_memory_footprint()/(1024*1024*1024)} GB')

memory footprint of model: 0.3178596496582031 GB


# Create or Load Lora Pretrained Model

In [ ]:

def find_all_linear_names(model):
    """
    找出所有全连接层，为所有全连接添加adapter
    """
    cls = bitsandbytes.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names:  # needed for 16-bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)


In [ ]:
# 找到所有需要插入adapter的全連接層
target_modules = find_all_linear_names(model)
print(target_modules)

['query_key_value', 'dense_h_to_4h', 'dense_4h_to_h', 'dense']


# Lora qlora 參數設定

In [ ]:
LORA_R = 64
LORA_ALPHA = 16
# LORA_R = 8
# LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# 初始化lora配置
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules, #新贈
)

model = peft.get_peft_model(model, peft_config)

In [ ]:
# 這裡我們只訓練了模型參數的0.16%！這個巨大的內存增益讓我們安心地微調模型，而不用擔心內存問題。
model.print_trainable_parameters()

trainable params: 25,165,824 || all params: 374,731,776 || trainable%: 6.71568989121435


In [ ]:
model.state_dict

<bound method Module.state_dict of PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 1024)
        (word_embeddings_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear4bit(
                in_features=1024, out_features=3072, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3072, bias=False)
                )
          

In [ ]:
# 新增不知用途??
# model.config.torch_dtype = torch.float32

In [ ]:
# 設定 `use_cache=True` is incompatible with gradient checkpointing.
model.config.use_cache = False

In [ ]:
model.state_dict

<bound method Module.state_dict of PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 1024)
        (word_embeddings_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0-23): 24 x BloomBlock(
            (input_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear4bit(
                in_features=1024, out_features=3072, bias=True
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=3072, bias=False)
                )
          

In [ ]:
model = torch.compile(model)

In [ ]:
device = torch.device("cuda")
model.cuda()

# Load dataset

In [ ]:
train_data = datasets.load_from_disk('train_dataset_belle_100k_YeungNLP/data_train/') #簡體中文
val_data = datasets.load_from_disk('train_dataset_belle_100k_YeungNLP/data_val/')

In [ ]:
train_data

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 48085
})

In [ ]:
data_collator = transformers.DataCollatorForSeq2Seq(
    tokenizer,
    return_tensors="pt",
    padding=True,
    pad_to_multiple_of=8,
    )

# Traing

In [ ]:
# optimized for RTX 3090 and A100. For larger GPUs, increase some of these?
MICRO_BATCH_SIZE = 36  # 影響內存
BATCH_SIZE = 80  # 似乎不太影響內存
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE

LEARNING_RATE = 2e-4  # the Karpathy constant
#LEARNING_RATE = 3e-4  # the Karpathy constant
# LEARNING_RATE = 2e-5 # ???


EVAL_BATCH_SIZE = 4

EPOCHS = 1  # we don't always need 3 tbh


OUTPUT_DIR = "my-checkpoints"


LOGGING_STEPS = 10
SAVE_STEPS = 10  # Reduce it to a smaler value like 512 if you want to save checkpoints
SAVE_TOTAL_LIMIT = 2

In [ ]:
training_args = TrainingArguments(
    overwrite_output_dir=True,
    load_best_model_at_end=False,
    optim='adamw_torch',
    save_strategy='steps',
    evaluation_strategy='steps',
    # save_strategy='epoch',
    # evaluation_strategy='epoch',
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    warmup_steps=500,
    weight_decay=0.1, #?? 多少適合?

    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,

    save_total_limit=SAVE_TOTAL_LIMIT,
    
    num_train_epochs=EPOCHS,
    
    learning_rate=LEARNING_RATE,

    fp16=True, # This is for GPU not for CPU 造成loss無法計算

    output_dir=OUTPUT_DIR,
    report_to='none',

)



# 
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    args=training_args,
    data_collator=data_collator,
)


In [ ]:
%%time
#trainer.train(resume_from_checkpoint=True) #
trainer.train()

You're using a BloomTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
460,2.371700,2.383409
470,2.362200,2.382716
480,2.386200,2.382385
490,2.387000,2.380789
500,2.392300,2.379747
510,2.342800,2.378778
520,2.375900,2.376383
530,2.343200,2.375016
540,2.404200,2.373840
550,2.344300,2.372998


CPU times: user 2h 56min 14s, sys: 1h 48s, total: 3h 57min 3s
Wall time: 3h 57min 13s


TrainOutput(global_step=668, training_loss=0.771169776688079, metrics={'train_runtime': 14225.5495, 'train_samples_per_second': 3.38, 'train_steps_per_second': 0.047, 'total_flos': 1.5333756799156224e+16, 'train_loss': 0.771169776688079, 'epoch': 1.0})

        430	2.383100	2.389674
        440	2.377100	2.390504
        450	2.422000	2.386701

In [ ]:
model.save_pretrained('my-qlora-model-1epochs')
#